# Pipeline de Machine Learning y Modelado Analítico
Este cuaderno contiene la extracción de datos desde **SQL Server** a través de vistas unificadas, el procesamiento de variables (Feature Engineering), el agrupamiento no supervisado (**K-Means**), el entrenamiento de modelos clasificadores supervisados (**Random Forest** y **XGBoost**), y el registro de experimentos en **MongoDB** para persistencia híbrida.

## 1. Importación de Librerías

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import joblib
from pymongo import MongoClient
import datetime

# Scikit-Learn y XGBoost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix, 
                             roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Asegurar directorio de gráficas
os.makedirs('graficas', exist_ok=True)
print('Librerías importadas.')

## 2. Extracción de Datos desde SQL Server
Consumimos la vista `vw_DatosUnificados` creada en la base de datos relacional.

In [ ]:
CONN_STR = 'mssql+pyodbc://LUIS/IncendiosForestalesEC?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
engine = create_engine(CONN_STR)

df = pd.read_sql('SELECT * FROM vw_DatosUnificados ORDER BY fecha, id_ciudad', con=engine)
print(f'Dataset cargado. Registros: {df.shape[0]}, Columnas: {df.shape[1]}')

## 3. Preprocesamiento e Ingeniería de Variables (Feature Engineering)
Creamos componentes temporales, codificamos variables categóricas e identificamos la estación seca en Ecuador.

In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'])
df['mes'] = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter
df['dia_anio'] = df['fecha'].dt.dayofyear

# Bandera de estación seca (Junio a Septiembre)
df['es_estacion_seca'] = df['mes'].isin([6, 7, 8, 9]).astype(int)

# Codificar ciudad
le_ciudad = LabelEncoder()
df['ciudad_cod'] = le_ciudad.fit_transform(df['ciudad_nombre'])

# Completar nulos residuales
df = df.ffill().bfill()
print('Feature engineering completado con éxito.')

## 4. Aprendizaje No Supervisado: Segmentación de Climas de Riesgo (K-Means)
Agrupamos los climas diarios utilizando K-Means. Evaluamos el número de clusters óptimo con el Método del Codo y el Coeficiente de Silueta.

In [ ]:
features_kmeans = ['temperatura_media', 'humedad_relativa', 'velocidad_viento', 'precipitacion', 'ndvi']
scaler_kmeans = StandardScaler()
X_kmeans = scaler_kmeans.fit_transform(df[features_kmeans])

inercias = []
siluetas = []
rango_k = range(2, 9)

# Muestra aleatoria para cálculo rápido de Silueta O(N^2)
sample_idx = np.random.RandomState(42).choice(len(X_kmeans), size=min(2000, len(X_kmeans)), replace=False)
X_kmeans_sample = X_kmeans[sample_idx]

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_kmeans)
    inercias.append(km.inertia_)
    
    score_sil = silhouette_score(X_kmeans_sample, labels[sample_idx], random_state=42)
    siluetas.append(score_sil)
    print(f'k={k} | Inercia: {km.inertia_:.2f} | Silueta: {score_sil:.4f}')

### Graficación de Curvas de Validación

In [ ]:
# Codo
plt.figure(figsize=(8, 3.5))
plt.plot(rango_k, inercias, marker='o', color='#1f77b4')
plt.title('Método del Codo')
plt.xlabel('K')
plt.ylabel('Inercia')
plt.grid(True)
plt.savefig('graficas/curva_codo.png', dpi=150)
plt.show()

# Silueta
plt.figure(figsize=(8, 3.5))
plt.plot(rango_k, siluetas, marker='o', color='#2ca02c')
plt.title('Coeficiente de Silueta')
plt.xlabel('K')
plt.ylabel('Silueta')
plt.grid(True)
plt.savefig('graficas/coeficiente_silueta.png', dpi=150)
plt.show()

### Ejecutar K-Means Definitivo con $K=3$

In [ ]:
kmeans_optimo = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans_optimo.fit_predict(X_kmeans)

# Ordenar clusters por temperatura media para establecer categorías estables:
# 0: Bajo Riesgo, 1: Medio Riesgo, 2: Alto Riesgo
cluster_temps = df.groupby('cluster')['temperatura_media'].mean().sort_values()
cluster_mapping = {old_label: new_label for new_label, old_label in enumerate(cluster_temps.index)}
df['cluster_riesgo'] = df['cluster'].map(cluster_mapping)

# Mostrar perfiles
perfiles = df.groupby('cluster_riesgo')[features_kmeans].mean()
perfiles['dias'] = df.groupby('cluster_riesgo').size()
perfiles['incendios'] = df.groupby('cluster_riesgo')['incendio_binario'].sum()
perfiles['tasa'] = df.groupby('cluster_riesgo')['incendio_binario'].mean()
print(perfiles)

## 5. Aprendizaje Supervisado: Clasificación de Incendios Diarios (Random Forest vs XGBoost)
Entrenamos dos modelos clasificadores para predecir si ocurrirá un foco de calor dado el clima diaria y vegetación. Usamos un split de 80/20 con `random_state=42`.

In [ ]:
features_supervised = [
    'ciudad_cod', 'ciudad_latitud', 'ciudad_longitud', 'altitud_msnm',
    'velocidad_viento', 'direccion_viento', 'temperatura_media', 'temperatura_max', 
    'temperatura_min', 'humedad_relativa', 'precipitacion', 'ndvi', 
    'mes', 'trimestre', 'dia_anio', 'es_estacion_seca'
]

X = df[features_supervised]
y = df['incendio_binario']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalamiento
scaler_supervised = StandardScaler()
X_train_scaled = scaler_supervised.fit_transform(X_train)
X_test_scaled = scaler_supervised.transform(X_test)
print('Split 80/20 completado.')

### Entrenamiento y Evaluación

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

def evaluar(y_true, y_pred, y_prob, name):
    metrics = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc_roc': float(roc_auc_score(y_true, y_prob))
    }
    print(f'Métricas {name}:')
    for k, v in metrics.items():
        print(f'  {k:10s}: {v:.4f}')
    return metrics

metrics_rf = evaluar(y_test, y_pred_rf, y_prob_rf, 'Random Forest')
metrics_xgb = evaluar(y_test, y_pred_xgb, y_prob_xgb, 'XGBoost')

best_name = 'XGBoost' if metrics_xgb['f1_score'] > metrics_rf['f1_score'] else 'Random Forest'
best_model = xgb_model if best_name == 'XGBoost' else rf_model
print(f'\n>> Mejor Modelo seleccionado: {best_name}')

### Curva ROC Comparativa e Importancia de Variables

In [ ]:
# Graficar ROC
plt.figure(figsize=(7, 5))
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
plt.plot(fpr_rf, tpr_rf, label=f'RF (AUC={metrics_rf["auc_roc"]:.3f})', color='#d62728')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGB (AUC={metrics_xgb["auc_roc"]:.3f})', color='#1f77b4')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC Comparativa')
plt.legend()
plt.savefig('graficas/curva_roc.png', dpi=150)
plt.show()

# Importancia de Variables del mejor modelo
importancias = best_model.feature_importances_
idx_sorted = np.argsort(importancias)[::-1]
plt.figure(figsize=(9, 5))
sns.barplot(x=importancias[idx_sorted], y=np.array(features_supervised)[idx_sorted], palette='viridis')
plt.title(f'Importancia de Variables ({best_name})')
plt.savefig('graficas/importancia_features.png', dpi=150)
plt.show()

## 6. Guardar Artefactos del Modelo
Exportamos el diccionario de modelos y escaladores a un archivo `.pkl` para ser consumido en Streamlit.

In [ ]:
model_artifacts = {
    'supervised_model': best_model,
    'supervised_model_name': best_name,
    'kmeans_model': kmeans_optimo,
    'scaler_supervised': scaler_supervised,
    'scaler_kmeans': scaler_kmeans,
    'le_ciudad': le_ciudad,
    'features_supervised': features_supervised,
    'features_kmeans': features_kmeans,
    'cluster_mapping': cluster_mapping
}
joblib.dump(model_artifacts, 'modelo_incendios.pkl')
print('Artefactos del modelo guardados en modelo_incendios.pkl.')

## 7. Exportación a JSON y Registro de Experimentos en MongoDB (Persistencia Híbrida)
Para cumplir con el diseño analítico del proyecto, formateamos las métricas como JSON y las guardamos localmente como archivo 'experimentos_ml.json', además de registrarlas en MongoDB local.

In [ ]:
doc_rf = {
    'proyecto': 'Incendios Forestales Ecuador 2012-2026',
    'fecha': datetime.datetime.now(),
    'algoritmo': 'Random Forest',
    'libreria': 'scikit-learn',
    'parametros': {
        'n_estimators': 100,
        'random_state': 42,
        'n_jobs': -1
    },
    'metricas': metrics_rf,
    'variables_entrada': features_supervised,
    'seleccionado': bool(best_name == 'Random Forest')
}

doc_xgb = {
    'proyecto': 'Incendios Forestales Ecuador 2012-2026',
    'fecha': datetime.datetime.now(),
    'algoritmo': 'XGBoost',
    'libreria': 'xgboost',
    'parametros': {
        'n_estimators': 100,
        'random_state': 42,
        'eval_metric': 'logloss',
        'n_jobs': -1
    },
    'metricas': metrics_xgb,
    'variables_entrada': features_supervised,
    'seleccionado': bool(best_name == 'XGBoost')
}

# A. Exportar a archivo JSON físico
try:
    import json
    def format_doc_for_json(d):
        d_copy = d.copy()
        if isinstance(d_copy['fecha'], datetime.datetime):
            d_copy['fecha'] = d_copy['fecha'].isoformat()
        return d_copy

    json_data = [format_doc_for_json(doc_rf), format_doc_for_json(doc_xgb)]
    with open('experimentos_ml.json', 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)
    print('- Archivo experimentos_ml.json guardado con éxito.')
except Exception as e:
    print(f'Error al exportar JSON: {e}')

# B. Persistencia en MongoDB
try:
    client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=2000)
    db = client['IncendiosForestales_ML']
    col = db['experimentos']
    col.delete_many({'proyecto': 'Incendios Forestales Ecuador 2012-2026'})
    col.insert_one(doc_rf)
    col.insert_one(doc_xgb)
    print('- Experimentos registrados en MongoDB (puerto 27017).')
    
    print('\nVerificación en MongoDB:')
    for doc in col.find({'proyecto': 'Incendios Forestales Ecuador 2012-2026'}):
        print(f"  * {doc['algoritmo']} | F1: {doc['metricas']['f1_score']:.4f} | Seleccionado: {doc['seleccionado']}")
except Exception as e:
    print(f'Error al conectar con MongoDB: {e}')